# Cypher L5 — docling escalation notebook

**When to use this notebook**: a PDF failed to extract via the local Cypher pipeline (L1 text-layer / L3 Tesseract OCR) because the source has no usable text layer AND L3's OCR output is too unstructured for the per-variant line parsers to recover rows from.

**What it does**:
1. Installs **docling** (IBM Research / LF AI & Data, MIT-licensed) on the Colab runtime.
2. Takes a single source PDF (your upload).
3. Runs docling's layout-aware pipeline — TableFormer for table structure, integrated OCR for scanned content, layout-analysis ML for reading order.
4. Returns a structured `DoclingDocument` and lets you inspect each `TableItem` programmatically — rows, columns, per-cell bounding boxes — *not* just a flat OCR blob.
5. Maps the recovered table cells onto your chosen Cypher variant's column schema and emits a CSV that drops into `research/results/by_pdf/` alongside the existing per-PDF outputs.

**Why L5 lives in a notebook, not the main pipeline**: docling depends on PyTorch + transformers + model downloads (~hundreds of MB on first run, 5–30 s per page on CPU). The local Cypher venv stays Python 3.9 / Pyodide-friendly, and the deploy stays light. L5 burst-runs on Colab for the fringe cases the main pipeline can't handle, identical to how the existing L4 PaddleOCR notebook works.

**Recon use** (first time you run this): point it at one of the image-only HT PDFs the corpus survey flagged and answer **does docling's TableItem output recover the row/column structure cleanly enough that a generic adapter could turn it into Cypher rows?** If yes → we wire L5 into the routing in `sheet_types/router.py`. If the recovery is poor on aviation-specific layouts → we stop here and reconsider.

**Recommended recon files** (image-only HT, varied operators):
* `HT Components Status-20180820.pdf` (3 pages, dense)
* `MSN 2333 - AVA HT Components Statement 16Jan2020.pdf` (1 page, AVA / Avianca)
* `MSN 3626 HTC 22-DEC-20.pdf` (6 pages)
* `MSN 1612 - AVIANCA HT Inventory 15Sep2015.pdf` (6 pages, AVIANCA)
* `HL7737(MSN 2397) HT Component Status_draft_2020.02.07.pdf` (8 pages, Korean Air HL)

## 1. Install

First-run installs docling + its model weights. Subsequent runs in the same Colab session are cached. If the GPU runtime is selected Colab will use it; otherwise everything works on CPU at ~5–30 s/page.

In [ ]:
%pip install --quiet docling pandas openpyxl
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', 'GPU — '+torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Upload PDF

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_name = next(iter(uploaded))
with open(pdf_name, 'wb') as fh:
    fh.write(uploaded[pdf_name])
print(f'Uploaded: {pdf_name}  ({len(uploaded[pdf_name]):,} bytes)')

## 3. Configure variant + page range

Picking the variant tells the adapter which Cypher column schema to map docling's recovered cells onto. Leave `PAGES = None` to process the whole PDF; restrict to specific pages for faster recon iteration. Set `MODE = 'recon'` for the first run so you get raw DoclingDocument inspection output as well as the row mapping; switch to `'production'` once the recovery looks right.

In [ ]:
VARIANT = 'AMOS HT'   # change to the variant you expect: 'AMOS HT' | 'MM_510' | 'TAP HT' | 'Standard OCCM' | etc.
PAGES   = None        # None = whole doc, or e.g. [1, 2, 3]
MODE    = 'recon'     # 'recon' (verbose, inspect raw output) | 'production' (just emit the CSV)

# Column schemas for the common variants — keep these aligned with
# sheet_types/<sheet>_variants/<variant>.py CANONICAL_COLUMNS.
VARIANT_COLUMNS = {
    'AMOS HT':       ['ATA', 'DESCRIPTION', 'PART_NUMBER', 'SERIAL_NUMBER', 'POS', 'INST_DATE', 'TSN', 'CSN'],
    'MM_510':        ['ATA', 'POSITION', 'PART_NUMBER', 'SERIAL_NUMBER', 'DESCRIPTION', 'INST_DATE', 'TASK', 'TASK_NUMBER', 'INTERVAL', 'REMAINING'],
    'TAP HT':        ['PART_NUMBER', 'SERIAL_NUMBER', 'DESCRIPTION', 'ATA', 'POSITION', 'INSTALL_DATE', 'FH'],
    'Standard OCCM': ['ATA', 'DESCRIPTION', 'FIN', 'PART_NUMBER', 'SERIAL_NUMBER', 'DATE', 'FH', 'FC'],
}
columns = VARIANT_COLUMNS.get(VARIANT, ['ATA', 'PART_NUMBER', 'SERIAL_NUMBER', 'POSITION', 'DESCRIPTION'])
print(f'Variant: {VARIANT}\nColumns: {columns}\nMode: {MODE}')

## 4. Run docling

docling's `DocumentConverter` runs the full pipeline (layout analysis → table-structure recognition → OCR) and returns a `ConversionResult` whose `.document` is a `DoclingDocument`.

First-run downloads the layout-analysis model (~100 MB) and the TableFormer model (~100 MB). They're cached after that.

In [ ]:
from docling.document_converter import DocumentConverter
import time

converter = DocumentConverter()
t0 = time.time()
result = converter.convert(pdf_name)
doc = result.document
elapsed = time.time() - t0
print(f'Converted in {elapsed:.1f} s')
print(f'Texts:    {len(doc.texts)}')
print(f'Tables:   {len(doc.tables)}')
print(f'Pictures: {len(doc.pictures)}')
print(f'Pages:    {len(doc.pages)}')

## 5. Recon — inspect what docling found

**This is the critical cell for the recon experiment.** What we want to see:

1. `len(doc.tables) > 0` — docling found tables at all
2. Each table has rows AND columns (not just one giant column)
3. Cell text matches what's visibly in the PDF (vs OCR garbage)
4. Cell bounding boxes are recoverable (the `prov` attribute) so we can sort cells back into row-order

If all four hold for the sample we'll know L5 is worth integrating.

In [ ]:
if MODE == 'recon':
    print(f'=== Found {len(doc.tables)} tables ===')
    for ti, table in enumerate(doc.tables):
        print(f'\n--- Table {ti} ---')
        # docling tables expose .data with a grid of cells
        try:
            n_rows = table.data.num_rows
            n_cols = table.data.num_cols
        except AttributeError:
            n_rows = n_cols = '?'
        print(f'  Shape: {n_rows} rows x {n_cols} cols')
        # Page provenance
        if table.prov:
            for p in table.prov[:1]:
                print(f'  Page: {p.page_no}, bbox: {p.bbox}')
        # Print first few rows of cell text
        try:
            grid = table.data.grid   # list[list[TableCell]]
            for ri, row in enumerate(grid[:5]):
                cells = [(c.text or '').strip().replace('\n', ' ')[:25] for c in row]
                print(f'  row {ri}: {cells}')
            if len(grid) > 5:
                print(f'  ... ({len(grid) - 5} more rows)')
        except Exception as e:
            # docling versions vary in how the grid is exposed; fall back to
            # the markdown export so we can at least eyeball the recovery.
            print(f'  (grid traversal failed: {e}; falling back to markdown)')
            print(table.export_to_markdown()[:600])
else:
    print(f'(recon output skipped — MODE={MODE})')

## 6. Adapter — TableItem → Cypher variant rows

Heuristics to map docling's recovered cells onto a Cypher variant's column schema:

1. Find the header row (first row whose cell text matches column-name keywords).
2. Index cells by detected column.
3. For each subsequent row, emit a dict keyed by variant columns.
4. Pipe each row through Cypher's `clean_record` if available locally (for now we just emit the raw dict; cleanup runs after download).

In [ ]:
import re

# Header-detection keywords per Cypher column. Lowercased substring match.
COLUMN_KEYWORDS = {
    'ATA':           ['ata', 'chapter'],
    'PART_NUMBER':   ['part no', 'part number', 'p/n', 'pn'],
    'SERIAL_NUMBER': ['serial no', 'serial number', 's/n', 'sn'],
    'POS':           ['pos.', 'position', 'fin'],
    'POSITION':      ['pos.', 'position', 'fin'],
    'DESCRIPTION':   ['description', 'desc', 'nomen', 'keyword'],
    'INST_DATE':     ['inst-date', 'install date', 'installed', 'installation'],
    'INSTALL_DATE':  ['inst-date', 'install date', 'installed', 'installation'],
    'DATE':          ['date'],
    'TSN':           ['tsn'],
    'CSN':           ['csn'],
    'FH':            ['fh', 'hours'],
    'FC':            ['fc', 'cycles', 'landings'],
    'FIN':           ['fin'],
    'TASK':          ['task', 'requirement'],
    'TASK_NUMBER':   ['taskcard', 'task no', 'card'],
    'INTERVAL':      ['interval'],
    'REMAINING':     ['remaining', 'to go', 'remain'],
}

def _cell_text(cell):
    return (cell.text or '').strip().replace('\n', ' ')

def _find_header_row(grid):
    """Return (header_idx, {column_name: col_idx}). Picks the first row whose
    cells contain enough column-keyword hits to look like a header."""
    for ri, row in enumerate(grid[:8]):
        texts = [_cell_text(c).lower() for c in row]
        mapping = {}
        for col_name, kws in COLUMN_KEYWORDS.items():
            for ci, t in enumerate(texts):
                if any(kw in t for kw in kws):
                    mapping.setdefault(col_name, ci)
                    break
        if len(mapping) >= 3:   # enough header tokens to be confident
            return ri, mapping
    return None, {}

def _table_to_rows(table, columns):
    try:
        grid = table.data.grid
    except AttributeError:
        return []
    header_idx, col_map = _find_header_row(grid)
    if header_idx is None:
        return []
    rows = []
    for row in grid[header_idx + 1:]:
        rec = {}
        for col_name in columns:
            ci = col_map.get(col_name)
            rec[col_name] = _cell_text(row[ci]) if ci is not None and ci < len(row) else ''
        # Page provenance from the table itself
        if table.prov:
            rec['_page'] = table.prov[0].page_no
        rec['_source'] = 'L5_docling'
        rows.append(rec)
    return rows

all_rows = []
for table in doc.tables:
    # Optional page filter
    if PAGES is not None and table.prov:
        if table.prov[0].page_no not in PAGES:
            continue
    all_rows.extend(_table_to_rows(table, columns))

print(f'Extracted {len(all_rows)} rows from {len(doc.tables)} table(s)')
if MODE == 'recon' and all_rows:
    print('\nFirst 5 rows:')
    for r in all_rows[:5]:
        print(' ', r)

## 7. Save + download CSV

In [ ]:
import pandas as pd, pathlib
df = pd.DataFrame(all_rows)
stem = pathlib.Path(pdf_name).stem.replace(' ', '_').replace('/', '_')
out_csv = f'{stem}_L5_docling.csv'
df.to_csv(out_csv, index=False)
print(f'Wrote {out_csv}  ({len(df)} rows, {len(df.columns)} cols)')
df.head(10)

In [ ]:
from google.colab import files as colab_files
colab_files.download(out_csv)

## 8. Merge into local results (do this on your machine)

After downloading the CSV, drop it into `research/results/by_pdf/` and re-run `tools/build_positions_db.py`. The build script picks up any `<stem>_<slug>.csv` file under by_pdf, so adopt the naming convention:

```
<file_stem>_l5_docling_<variant_slug>.csv
```

Until L5 has its own slot in `tools/build_positions_db.py:POSITION_MAP`, you can manually copy the rows into the matching variant's existing CSV (de-duplicating by part_number + serial_number + position).

### Recon decision record

After running this notebook on the recommended sample, note:

* **Tables found**: did docling detect ≥1 table on every page that visually has one?
* **Row count vs. ground truth**: roughly how many rows did docling recover vs. what you can see in the PDF?
* **Column mapping**: did the header-row detector find at least 3 Cypher columns automatically?
* **OCR quality**: are PN / SN strings recognisable, or full of `0`↔`O` / `1`↔`l` confusion?
* **Time per page**: CPU runtime — feasible to escalate ~133 files via Colab on demand?

Capture findings in `docs/TODO.md` under the L5 entry and decide on integration.